# E1·E2 — ResNet-20 FP32 재현과 INT8 정확도 하락 측정

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsi03048/resnet20-accel/blob/main/notebooks/E1_E2_resnet20_reference.ipynb)

| 단계 | 하는 일 | 산출물 |
| --- | --- | --- |
| E1 | 공개 checkpoint(학습 없음)로 CIFAR-10 Test 10,000장 직접 평가, sample 0 의 경계별 Tensor 저장 | `results/e1_fp32.json`, `golden/e1_sample0/` |
| E2 | BN Folding → W8 / W8A8 fake-quant 정확도 측정 → INT8 채택 근거 | `results/e2_ptq.json` |

- 런타임: **GPU 권장**(런타임 → 런타임 유형 변경 → T4). CPU 에서도 동작한다.
- 모델·checkpoint·전처리는 `model/common.py` 에 고정돼 있다. 바꾸면 결과를 새 버전으로 기록한다.
- E2 는 **INT8 을 쓸지 정하는 근거**를 만든다. 아직 RTL 용 bit-exact 정수 Reference 가 아니다.

In [ ]:
# 0. 저장소와 참고 구현 준비
import os, sys
if not os.path.isdir("resnet20-accel"):
    !git clone -q https://github.com/tsi03048/resnet20-accel.git
%cd resnet20-accel
!git rev-parse --short HEAD
sys.path.insert(0, os.getcwd())

import torch
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.__version__)

In [ ]:
# 1. 모델·데이터 (checkpoint hash 와 commit 을 확인하고 strict load)
from model.common import fetch_reference, load_resnet20, cifar10
ref = fetch_reference()
model, info = load_resnet20(ref, device)
test_set = cifar10("data", train=False)
train_set = cifar10("data", train=True)   # E2 Calibration 전용
info

In [ ]:
# 2. E1 — FP32 직접 평가 + Golden Tensor 저장
from model.e1_fp32 import run_e1
e1, manifest, fp32_preds = run_e1(model, info, test_set, device)
print(f"FP32 Top-1 (직접 평가): {e1['fp32_top1']:.2f}%  ({e1['correct']}/{e1['test_images']})")
print(f"checkpoint 저장값 best_prec1: {e1['stored_best_prec1_in_checkpoint']:.2f}%  ← 참고만")
print("sample 0:", manifest["label_name"], "→", manifest["pred_name"])
e1["per_class_top1"]

## E2 — INT8 을 쓸지 정하기

`ACCEPT_DROP_PP` 는 **팀이 정하는 허용 하락 폭(%p)** 이다. 값을 바꿔도 측정 결과는 같고, 판단 표시만 바뀐다.

In [ ]:
ACCEPT_DROP_PP = 1.0   # 팀 결정값으로 수정

from quant.e2_ptq import run_e2
e2 = run_e2(model, test_set, train_set, device, fp32_preds, e1["fp32_top1"],
            accept_drop_pp=ACCEPT_DROP_PP)
print("BN Folding 최대 logit 차이(64장):", e2["folding_max_abs_logit_diff_64imgs"])
import pandas as pd
pd.DataFrame(e2["results"])

In [ ]:
d = e2["decision_input"]
print(f"허용 하락 폭: {d['accept_drop_pp']} %p")
print(f"가장 좋은 INT8 설정: {d['best_int8_config']}  (하락 {d['best_int8_drop_pp']} %p)")
print("→ INT8 채택 권고" if d["recommend_int8"] else "→ INT8 보류 (FP32/다른 정밀도 검토)")

## 결과 공유

아래 셀은 `results/`·`golden/` 을 zip 으로 묶는다. 다운로드해 Pull Request 로 올리거나, 담당자에게 전달한다.
Notion 의 E1·E2 페이지에는 **Top-1, 하락 폭, commit, 실행자, 날짜**를 기록한다.

In [ ]:
!git rev-parse HEAD > results/code_commit.txt
!zip -qr e1_e2_outputs.zip results golden
try:
    from google.colab import files
    files.download("e1_e2_outputs.zip")
except ImportError:
    print("Colab 이 아니면 e1_e2_outputs.zip 을 직접 가져간다.")